# PDCP · N07 — LiteJudge training data (CPU)
**Notebook 7 of 13.** Kaggle: Accelerator **None (CPU)** · Internet **On** · **Save & Run All (Commit)**. Expected ≈ 1.5–2.5 h. **No inputs needed.**

**Purpose.** Training/validation/test pairs for LiteJudge (contribution C4) on **new graphs that never appear in the VLM evaluation**. Two kinds of pairs:
1. **natural** — pairs of ordinary natural drawings of the same graph (stress 0.30–0.50). Here stress is confounded with the visual cues, exactly like ordinary layout data.
2. **pdsg** — decoupled pairs from the PDSG generator. Following the E1/E2 findings, **four** cues are now controlled relative to the partner drawing: node uniformity (NU), edge-length deviation (ELD), crossings (XR) **and mean edge length (MEL)**, with 7 conditions (congruent, neutral, conflict on each cue, conflict on all) × Δstress {0, .03, .06, .10, .15, .20}.

N11 trains LiteJudge with 0–50 % pdsg pairs mixed into natural pairs and tests whether decoupled training removes the cue shortcuts (H7).

**Graph split (by graph, no leakage):** 24 graphs per family set → 18 train · 3 val · 3 test; graph seeds are disjoint from the evaluation graphs of N03 (checked below).

**Outputs** (`/kaggle/working/n07_output/`): `pdcp_utils.py` v0.1.3, `lj_drawings.parquet`, `lj_pairs.parquet`, `images/384/*.png`, `n07_report.json`, figures, `MANIFEST.txt`.

In [1]:
# ==== SETTINGS ====
OUTPUT_DIR = "/kaggle/working/n07_output"
SEED = 2026                       # used only to rebuild the N03 evaluation graphs for the disjointness check
LJ_SEED = 9_000_000               # base seed for LiteJudge graphs (far from evaluation seeds)
FAMILY_SETS = [("BA", 50), ("BA", 100), ("ER", 25), ("ER", 50), ("ER", 100), ("GRID", 49), ("SBM", 50), ("SBM", 100), ("TREE", 50), ("WS", 50)]
GRAPHS_PER_SET = 24
SPLIT = {"train": range(0, 18), "val": range(18, 21), "test": range(21, 24)}
STRESS_LEVELS = [0.30, 0.35, 0.40, 0.45, 0.50]
DSTRESS = [0.0, 0.03, 0.06, 0.10, 0.15, 0.20]
CUES = ["nu", "eld", "xr", "mel"]
MARGIN = {"nu": 0.05, "eld": 0.05, "xr": 0.04, "mel": 0.04}
CONDS = {"congruent": (-1, -1, -1, -1), "neutral": (0, 0, 0, 0), "conflict_nu": (1, 0, 0, 0), "conflict_eld": (0, 1, 0, 0),
         "conflict_xr": (0, 0, 1, 0), "conflict_mel": (0, 0, 0, 1), "conflict_all": (1, 1, 1, 1)}
STRESS_FLOOR = 0.295
RES = 384
N_JOBS = 4
MAX_HOURS = 10.0
FAST = False

In [2]:
import os, sys, json, time, itertools, subprocess, importlib, shutil
T0 = time.time()
os.makedirs(f"{OUTPUT_DIR}/images/{RES}", exist_ok=True)
for p in ["joblib", "pyarrow"]:
    try: importlib.import_module(p)
    except ImportError: subprocess.run([sys.executable, "-m", "pip", "install", "-q", p], check=True)
import numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg"); import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from scipy.stats import pearsonr, spearmanr
if FAST:
    FAMILY_SETS = [("ER", 25), ("TREE", 50)]; GRAPHS_PER_SET = 4; SPLIT = {"train": range(0, 2), "val": range(2, 3), "test": range(3, 4)}
    DSTRESS = [0.0, 0.06, 0.15]
def hours(): return (time.time() - T0) / 3600

## 1. Shared module v0.1.3

In [3]:
%%writefile /kaggle/working/n07_output/pdcp_utils.py
"""
pdcp_utils.py  —  shared code for the PDCP project (Proxy-Decoupled Counterfactual Probing)
Version: 0.1.0  (written by N01, 2026-11)

Contents
  1. Graph families            make_graph(family, n, seed)
  2. Exact metrics             ksm, node_uniformity, edge_length_deviation, crossing_ratio, all_metrics
  3. Differentiable surrogates (torch)  used only inside the optimiser
  4. PDSG Stage 1              pdsg_optimise(G, targets, ...)
  5. Rendering                 render_png(G, X, path, ...)
  6. Parsing model output      parse_answer(text)
  7. Statistics helpers        bootstrap_ci(...)

All coordinates X are numpy arrays of shape (n, 2). Node order = sorted(G.nodes()) after relabel to 0..n-1.
"""
from __future__ import annotations

import json
import math
import re
import time
from dataclasses import dataclass, field

import networkx as nx
import numpy as np

__version__ = "0.1.3"

# ----------------------------------------------------------------------------
# 1. Graph families
# ----------------------------------------------------------------------------
FAMILY_SIZES = {
    "ER": [10, 25, 50, 100],
    "BA": [25, 50, 100],
    "SBM": [25, 50, 100],
    "WS": [25, 50],
    "GRID": [16, 25, 49, 100],
    "TREE": [25, 50, 100],
}


def _connected_or_retry(fn, seed, max_tries=200):
    for t in range(max_tries):
        G = fn(seed + 1000 * t)
        if nx.is_connected(G):
            return G
    raise RuntimeError("could not sample a connected graph")


def make_graph(family: str, n: int, seed: int) -> nx.Graph:
    """Return a connected simple graph with nodes relabelled 0..n-1."""
    family = str(family).upper(); n = int(n); seed = int(seed)
    if family == "ER":
        p = 3.5 / (n - 1)  # expected average degree 3.5
        G = _connected_or_retry(lambda s: nx.gnp_random_graph(n, p, seed=s), seed)
    elif family == "BA":
        G = nx.barabasi_albert_graph(n, 2, seed=seed)
    elif family == "SBM":
        k = {25: 2, 50: 3, 100: 4}.get(n, max(2, n // 25))
        p_in = {25: 0.40, 50: 0.25, 100: 0.15}.get(n, 0.2)
        sizes = [n // k] * k
        sizes[-1] += n - sum(sizes)
        P = [[p_in if i == j else 0.02 for j in range(k)] for i in range(k)]
        G = _connected_or_retry(lambda s: nx.stochastic_block_model(sizes, P, seed=s), seed)
        G = nx.Graph(G)
    elif family == "WS":
        G = nx.connected_watts_strogatz_graph(n, 4, 0.1, seed=seed)
    elif family == "GRID":
        a = int(round(math.sqrt(n)))
        G = nx.grid_2d_graph(a, a)
    elif family == "TREE":
        try:
            G = nx.random_labeled_tree(n, seed=seed)
        except AttributeError:  # older networkx
            G = nx.random_tree(n, seed=seed)
    else:
        raise ValueError(f"unknown family {family}")
    G = nx.convert_node_labels_to_integers(G, ordering="sorted")
    G.remove_edges_from(nx.selfloop_edges(G))
    return G


def graph_arrays(G: nx.Graph):
    """Edge index array (m,2), shortest-path matrix D (n,n), and index arrays for i<j pairs."""
    n = G.number_of_nodes()
    E = np.array(sorted((min(u, v), max(u, v)) for u, v in G.edges()), dtype=np.int64)
    D = np.zeros((n, n), dtype=np.float64)
    for s, lengths in nx.all_pairs_shortest_path_length(G):
        for t, d in lengths.items():
            D[s, t] = d
    iu, ju = np.triu_indices(n, k=1)
    return E, D, iu, ju


def nonadjacent_edge_pairs(E: np.ndarray):
    """All pairs of edges that share no endpoint (the pairs that can cross)."""
    m = len(E)
    a, b = np.triu_indices(m, k=1)
    e1, e2 = E[a], E[b]
    share = (e1[:, 0] == e2[:, 0]) | (e1[:, 0] == e2[:, 1]) | (e1[:, 1] == e2[:, 0]) | (e1[:, 1] == e2[:, 1])
    return a[~share], b[~share]


# ----------------------------------------------------------------------------
# 2. Exact metrics
# ----------------------------------------------------------------------------
def normalise(X: np.ndarray) -> np.ndarray:
    """Uniform scale into the unit box, aspect ratio preserved, centred."""
    X = np.asarray(X, dtype=np.float64)
    X = X - X.min(0)
    s = X.max()
    if s <= 0:
        return np.full_like(X, 0.5)
    X = X / s
    X = X + (1.0 - X.max(0)) / 2.0
    return X


def _pav(y: np.ndarray, w: np.ndarray | None = None) -> np.ndarray:
    """Pool-adjacent-violators: isotonic (non-decreasing) fit of y in given order."""
    y = np.asarray(y, dtype=np.float64)
    w = np.ones_like(y) if w is None else np.asarray(w, dtype=np.float64)
    vals, wts, cnt = [], [], []
    for yi, wi in zip(y, w):
        vals.append(yi); wts.append(wi); cnt.append(1)
        while len(vals) > 1 and vals[-2] > vals[-1]:
            v2, w2, c2 = vals.pop(), wts.pop(), cnt.pop()
            v1, w1, c1 = vals.pop(), wts.pop(), cnt.pop()
            wt = w1 + w2
            vals.append((v1 * w1 + v2 * w2) / wt); wts.append(wt); cnt.append(c1 + c2)
    return np.repeat(vals, cnt)


def ksm(X: np.ndarray, D: np.ndarray, iu=None, ju=None, ties: str = "secondary") -> float:
    """
    Kruskal (non-metric) stress in [0,1]; dhat = isotonic regression of e on d.
    ties='secondary' (default): all pairs with equal graph distance get the same dhat
        -> identical to sklearn.isotonic.IsotonicRegression, the convention used by Mooney et al.
    ties='primary': Kruskal's primary approach (ties may be broken); gives lower values. Kept for reference only.
    """
    n = len(X)
    if iu is None:
        iu, ju = np.triu_indices(n, k=1)
    d = D[iu, ju]
    e = np.linalg.norm(X[iu] - X[ju], axis=1)
    if ties == "primary":
        order = np.lexsort((e, d))
        dhat = np.empty_like(e)
        dhat[order] = _pav(e[order])
    else:
        levels, inv, cnt = np.unique(d, return_inverse=True, return_counts=True)
        means = np.bincount(inv, weights=e) / cnt
        fit = _pav(means, cnt)
        dhat = fit[inv]
    num = np.sum((e - dhat) ** 2)
    den = np.sum(e ** 2)
    return float(np.sqrt(num / den)) if den > 0 else 0.0


def node_uniformity(X: np.ndarray) -> float:
    """0 = nodes spread uniformly over a ceil(sqrt n)^2 grid on the drawing's bounding box, 1 = all in one cell."""
    n = len(X)
    g = int(math.ceil(math.sqrt(n)))
    lo, hi = X.min(0), X.max(0)
    span = np.where(hi - lo > 0, hi - lo, 1.0)
    U = (X - lo) / span
    idx = np.minimum((U * g).astype(int), g - 1)
    counts = np.zeros((g, g))
    np.add.at(counts, (idx[:, 0], idx[:, 1]), 1)
    cbar = n / g ** 2
    num = np.sqrt(np.sum((counts - cbar) ** 2))
    den = np.sqrt((n - cbar) ** 2 + (g ** 2 - 1) * cbar ** 2)
    return float(num / den)


def edge_length_deviation(X: np.ndarray, E: np.ndarray) -> float:
    """Coefficient of variation of edge lengths (scale invariant)."""
    L = np.linalg.norm(X[E[:, 0]] - X[E[:, 1]], axis=1)
    mu = L.mean()
    return float(L.std() / mu) if mu > 0 else 0.0


def mean_edge_length(X: np.ndarray, E: np.ndarray) -> float:
    """Mean edge length after uniform normalisation to the unit box (edge-length scale cue; added v0.1.3)."""
    X = normalise(X)
    return float(np.linalg.norm(X[E[:, 0]] - X[E[:, 1]], axis=1).mean())


def _orient(p, q, r):
    return (q[..., 0] - p[..., 0]) * (r[..., 1] - p[..., 1]) - (q[..., 1] - p[..., 1]) * (r[..., 0] - p[..., 0])


def crossing_count(X: np.ndarray, E: np.ndarray, pairs=None) -> int:
    if pairs is None:
        pairs = nonadjacent_edge_pairs(E)
    a, b = pairs
    if len(a) == 0:
        return 0
    P1, P2 = X[E[a, 0]], X[E[a, 1]]
    Q1, Q2 = X[E[b, 0]], X[E[b, 1]]
    o1, o2 = _orient(P1, P2, Q1), _orient(P1, P2, Q2)
    o3, o4 = _orient(Q1, Q2, P1), _orient(Q1, Q2, P2)
    cross = (o1 * o2 < 0) & (o3 * o4 < 0)  # proper crossings only; collinear touches ignored
    return int(cross.sum())


def crossing_ratio(X: np.ndarray, E: np.ndarray, pairs=None) -> float:
    if pairs is None:
        pairs = nonadjacent_edge_pairs(E)
    cmax = len(pairs[0])
    return crossing_count(X, E, pairs) / cmax if cmax > 0 else 0.0


@dataclass
class GraphCache:
    """Pre-computed arrays for one graph, reused across many drawings."""
    G: nx.Graph
    E: np.ndarray = field(init=False)
    D: np.ndarray = field(init=False)
    iu: np.ndarray = field(init=False)
    ju: np.ndarray = field(init=False)
    pairs: tuple = field(init=False)

    def __post_init__(self):
        self.E, self.D, self.iu, self.ju = graph_arrays(self.G)
        self.pairs = nonadjacent_edge_pairs(self.E)

    @property
    def n(self):
        return self.G.number_of_nodes()


def realism_metrics(X: np.ndarray) -> dict:
    """Shape diagnostics used to reject degenerate drawings (not optimisation targets).
    aspect   = short side / long side of the bounding box (1 = square)
    nn_max   = largest nearest-neighbour distance * sqrt(n)   (large = isolated outlier node)
    nn_min   = smallest pairwise distance * sqrt(n)            (small = overlapping nodes)
    dense    = fraction of nodes inside the densest cell of a 4x4 grid (large = clump)"""
    X = normalise(X)
    n = len(X)
    span = X.max(0) - X.min(0)
    aspect = float(span.min() / span.max()) if span.max() > 0 else 0.0
    Dm = np.linalg.norm(X[:, None] - X[None], axis=-1) + np.eye(n) * 10
    nn = Dm.min(1)
    lo, hi = X.min(0), X.max(0)
    idx = np.minimum(((X - lo) / np.where(hi - lo > 0, hi - lo, 1) * 4).astype(int), 3)
    counts = np.zeros((4, 4)); np.add.at(counts, (idx[:, 0], idx[:, 1]), 1)
    return {"aspect": aspect, "nn_max": float(nn.max() * math.sqrt(n)),
            "nn_min": float(nn.min() * math.sqrt(n)), "dense": float(counts.max() / n)}


REALISM_LIMITS = {"aspect_min": 0.70, "nn_max_max": 2.6, "nn_min_min": 0.20, "dense_max": 0.35}


def realism_ok(r: dict, lim: dict = REALISM_LIMITS) -> bool:
    return (r["aspect"] >= lim["aspect_min"] and r["nn_max"] <= lim["nn_max_max"]
            and r["nn_min"] >= lim["nn_min_min"] and r["dense"] <= lim["dense_max"])


def all_metrics(X: np.ndarray, gc: GraphCache, with_realism: bool = False) -> dict:
    X = normalise(X)
    out = {
        "ksm": ksm(X, gc.D, gc.iu, gc.ju),
        "nu": node_uniformity(X),
        "eld": edge_length_deviation(X, gc.E),
        "xr": crossing_ratio(X, gc.E, gc.pairs),
        "mel": mean_edge_length(X, gc.E),
    }
    if with_realism:
        rm = realism_metrics(X)
        out.update(rm)
        out["realistic"] = realism_ok(rm)
    return out


# ----------------------------------------------------------------------------
# 3. Differentiable surrogates (torch)
# ----------------------------------------------------------------------------
def _torch():
    import torch
    return torch


def t_normalise(X):
    """Differentiable uniform-scale normalisation to the unit box."""
    X = X - X.min(0).values
    return X / (X.max() + 1e-9)


def sur_stress(X, d, iu, ju):
    """Linear-scale stress: sqrt(sum (e - a d)^2 / sum e^2), a = <e,d>/<d,d>. Upper bound of KSM."""
    torch = _torch()
    e = torch.linalg.norm(X[iu] - X[ju], dim=1)
    a = (e * d).sum() / (d * d).sum()
    return torch.sqrt(((e - a * d) ** 2).sum() / (e * e).sum() + 1e-12)


def sur_nu(X, n):
    """Soft-histogram node uniformity on the per-axis bounding box grid."""
    torch = _torch()
    g = int(math.ceil(math.sqrt(n)))
    lo = X.min(0).values
    span = (X.max(0).values - lo).clamp_min(1e-9)
    U = (X - lo) / span
    centres = (torch.arange(g, dtype=X.dtype) + 0.5) / g
    cx, cy = torch.meshgrid(centres, centres, indexing="ij")
    C = torch.stack([cx.reshape(-1), cy.reshape(-1)], 1)          # (g^2, 2)
    sigma = 0.5 / g
    d2 = ((U[:, None, :] - C[None, :, :]) ** 2).sum(-1)           # (n, g^2)
    W = torch.softmax(-d2 / (2 * sigma ** 2), dim=1)
    counts = W.sum(0)
    cbar = n / g ** 2
    num = torch.sqrt(((counts - cbar) ** 2).sum() + 1e-12)
    den = math.sqrt((n - cbar) ** 2 + (g ** 2 - 1) * cbar ** 2)
    return num / den


def sur_eld(X, E):
    torch = _torch()
    L = torch.linalg.norm(X[E[:, 0]] - X[E[:, 1]], dim=1)
    return L.std(unbiased=False) / (L.mean() + 1e-12)


def sur_mel(X, E):
    torch = _torch()
    return torch.linalg.norm(X[E[:, 0]] - X[E[:, 1]], dim=1).mean()


def sur_xr(X, E, pa, pb, k=50.0):
    """Smooth crossing ratio. Orientation tests are normalised (sine of angle) so k is scale free."""
    torch = _torch()
    if len(pa) == 0:
        return X.sum() * 0.0
    P1, P2 = X[E[pa, 0]], X[E[pa, 1]]
    Q1, Q2 = X[E[pb, 0]], X[E[pb, 1]]

    def o(p, q, r):
        v1, v2 = q - p, r - p
        cr = v1[:, 0] * v2[:, 1] - v1[:, 1] * v2[:, 0]
        return cr / (torch.linalg.norm(v1, dim=1) * torch.linalg.norm(v2, dim=1) + 1e-9)

    s1 = -o(P1, P2, Q1) * o(P1, P2, Q2)
    s2 = -o(Q1, Q2, P1) * o(Q1, Q2, P2)
    soft = torch.sigmoid(k * s1) * torch.sigmoid(k * s2)
    return soft.sum() / len(pa)


def sur_realism(X, r_max, lam_aspect=1.0):
    """Penalise outliers (nearest-neighbour distance > r_max) and elongated drawings (aspect ratio far from 1).
    Without this, the optimiser reaches high stress or a target NU through degenerate shapes:
    a few far-away nodes stretching the bounding box, or a thin strip."""
    torch = _torch()
    n = X.shape[0]
    D = torch.cdist(X, X) + torch.eye(n, dtype=X.dtype) * 10.0
    nn = D.min(1).values
    outlier = (torch.relu(nn - r_max) ** 2).sum() / (r_max ** 2)
    span = X.max(0).values - X.min(0).values
    aspect = (span[0] - span[1]) ** 2 / (span.max() ** 2 + 1e-12)
    return outlier + lam_aspect * aspect * n


def sur_overlap(X, r_min=0.03):
    torch = _torch()
    n = X.shape[0]
    iu, ju = torch.triu_indices(n, n, offset=1)
    dist = torch.linalg.norm(X[iu] - X[ju], dim=1)
    return (torch.relu(r_min - dist) ** 2).sum()


# ----------------------------------------------------------------------------
# 4. PDSG Stage 1 — optimise one drawing toward target metric values
# ----------------------------------------------------------------------------
DEFAULT_TOL = {"ksm": 0.01, "nu": 0.02, "eld": 0.02, "xr": 0.01, "mel": 0.01}


@dataclass
class PDSGResult:
    X: np.ndarray
    metrics: dict
    converged: bool
    iters: int
    restarts: int
    seconds: float
    errors: dict


def initial_layout(G, low_stress: bool, rng: np.random.Generator):
    n = G.number_of_nodes()
    if low_stress:
        pos = nx.kamada_kawai_layout(G)
        X = np.array([pos[i] for i in range(n)])
        X = normalise(X) + rng.normal(0, 0.02, size=(n, 2))
    else:
        X = rng.uniform(0, 1, size=(n, 2))
    th = rng.uniform(0, 2 * math.pi)
    R = np.array([[math.cos(th), -math.sin(th)], [math.sin(th), math.cos(th)]])
    return normalise(X @ R.T)


def pdsg_optimise(gc: GraphCache, targets: dict, tol: dict | None = None, X0: np.ndarray | None = None,
                  max_steps: int = 2000, max_restarts: int = 5, lr: float = 0.05, check_every: int = 50,
                  k_xr: float = 50.0, lam_overlap: float = 1.0, spacing_c: float = 0.45, r_min: float | None = None,
                  outlier_c: float = 2.0, lam_realism: float = 1.0,
                  seed: int = 0, verbose: bool = False) -> PDSGResult:
    """
    targets: dict with any of keys 'ksm','nu','eld','xr' -> float target, or None/absent = free.
    Loss = sum_p ((sur_p - t~_p)/tol_p)^2 + lam_overlap * overlap.
    Spacing: overlap penalty with r_min = spacing_c / sqrt(n) (unit box) stops the optimiser reaching
    high stress by collapsing nodes into a clump (a degenerate, trivially detectable drawing).
    Realism: nearest-neighbour distance capped at outlier_c / sqrt(n); aspect ratio pulled toward 1.
    Outer correction: every `check_every` steps the surrogate target t~_p is shifted by (target - exact),
    so the optimiser converges on the EXACT metric even though it differentiates a surrogate.
    """
    torch = _torch()
    tol = {**DEFAULT_TOL, **(tol or {})}
    active = {k: v for k, v in targets.items() if v is not None}
    seed = int(seed)
    rng = np.random.default_rng(seed)
    torch.manual_seed(seed)
    t0 = time.time()

    E_t = torch.as_tensor(gc.E, dtype=torch.long)
    d_t = torch.as_tensor(gc.D[gc.iu, gc.ju], dtype=torch.float64)
    iu_t = torch.as_tensor(gc.iu, dtype=torch.long)
    ju_t = torch.as_tensor(gc.ju, dtype=torch.long)
    pa_t = torch.as_tensor(gc.pairs[0], dtype=torch.long)
    pb_t = torch.as_tensor(gc.pairs[1], dtype=torch.long)
    n = gc.n
    if r_min is None:
        r_min = spacing_c / math.sqrt(n)
    r_max = outlier_c / math.sqrt(n)

    def surrogates(Xt):
        Xn = t_normalise(Xt)
        out = {}
        if "ksm" in active:
            out["ksm"] = sur_stress(Xn, d_t, iu_t, ju_t)
        if "nu" in active:
            out["nu"] = sur_nu(Xn, n)
        if "eld" in active:
            out["eld"] = sur_eld(Xn, E_t)
        if "xr" in active:
            out["xr"] = sur_xr(Xn, E_t, pa_t, pb_t, k_xr)
        if "mel" in active:
            out["mel"] = sur_mel(Xn, E_t)
        return out, Xn

    if X0 is None:
        X0 = initial_layout(gc.G, low_stress=(active.get("ksm", 0.5) <= 0.55), rng=rng)
    best = None
    total_iters = 0
    X_start = np.array(X0, dtype=np.float64)

    for restart in range(max_restarts + 1):
        Xt = torch.tensor(X_start, dtype=torch.float64, requires_grad=True)
        opt = torch.optim.Adam([Xt], lr=lr, betas=(0.9, 0.99))
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max_steps, eta_min=lr * 0.1)
        sur_target = dict(active)
        for step in range(1, max_steps + 1):
            opt.zero_grad()
            sur, Xn = surrogates(Xt)
            loss = sum(((sur[k] - sur_target[k]) / tol[k]) ** 2 for k in sur)
            loss = loss + lam_overlap * sur_overlap(Xn, r_min) / (r_min ** 2) + lam_realism * sur_realism(Xn, r_max)
            loss.backward()
            opt.step()
            sched.step()
            total_iters += 1
            if step % check_every == 0 or step == max_steps:
                Xnp = normalise(Xt.detach().numpy())
                m = all_metrics(Xnp, gc)
                err = {k: m[k] - active[k] for k in active}
                score = sum(abs(err[k]) / tol[k] for k in active)
                if best is None or score < best[0]:
                    best = (score, Xnp.copy(), m, err)
                if all(abs(err[k]) < tol[k] for k in active):
                    return PDSGResult(_random_flip(Xnp, rng), m, True, total_iters, restart, time.time() - t0, err)
                for k in active:  # outer correction on surrogate targets
                    sur_target[k] = sur_target[k] - 0.8 * err[k]
                if verbose and step % 500 == 0:
                    print(restart, step, {k: round(v, 4) for k, v in m.items()})
        X_start = best[1] + rng.normal(0, 0.1, size=best[1].shape)

    _, Xb, m, err = best
    return PDSGResult(_random_flip(Xb, rng), m, False, total_iters, max_restarts, time.time() - t0, err)


def _random_flip(X, rng):
    """Random reflection per axis: preserves every metric (incl. NU), unlike arbitrary rotation."""
    X = X.copy()
    if rng.random() < 0.5:
        X[:, 0] = 1 - X[:, 0]
    if rng.random() < 0.5:
        X[:, 1] = 1 - X[:, 1]
    if rng.random() < 0.5:
        X = X[:, ::-1].copy()
    return X


# ----------------------------------------------------------------------------
# 5. Rendering
# ----------------------------------------------------------------------------
RENDER_DEFAULT = dict(res=768, pad=24, node_r=6.0, edge_w=1.5, node_fill="#333333", edge_col="#000000",
                      bg="#FFFFFF", supersample=4, antialias=True)


def render_png(gc: GraphCache, X: np.ndarray, path: str | None = None, **kw):
    """Draw edges then nodes. Returns a PIL image; saves to `path` if given."""
    from PIL import Image, ImageDraw
    o = {**RENDER_DEFAULT, **kw}
    ss = o["supersample"] if o["antialias"] else 1
    R = o["res"] * ss
    pad = o["pad"] * ss
    Xn = normalise(X)
    P = pad + Xn * (R - 2 * pad)
    P[:, 1] = R - P[:, 1]  # y up
    img = Image.new("RGB", (R, R), o["bg"])
    dr = ImageDraw.Draw(img)
    ew = max(1, int(round(o["edge_w"] * ss)))
    for u, v in gc.E:
        dr.line([tuple(P[u]), tuple(P[v])], fill=o["edge_col"], width=ew)
    r = o["node_r"] * ss
    for x, y in P:
        dr.ellipse([x - r, y - r, x + r, y + r], fill=o["node_fill"])
    if ss > 1:
        img = img.resize((o["res"], o["res"]), Image.LANCZOS)
    if path:
        img.save(path, optimize=True)
    return img


# ----------------------------------------------------------------------------
# 6. Parsing model output
# ----------------------------------------------------------------------------
_ANS_RE = re.compile(r'answer"?\s*(?:is)?\s*[:=]?\s*"?\s*(image\s*)?(1|2|same|no clear difference)\b', re.I)


def parse_answer(text: str) -> dict:
    """Return {'answer': '1'|'2'|'same'|None, 'confidence': int|None, 'reason': str|None, 'parse_ok': bool}."""
    out = {"answer": None, "confidence": None, "reason": None, "parse_ok": False}
    if not text:
        return out
    m = re.search(r"\{.*\}", text, re.S)
    if m:
        try:
            obj = json.loads(m.group(0))
            a = str(obj.get("answer", "")).strip().lower().replace("image", "").strip()
            if a in ("1", "2"):
                out["answer"] = a
            elif a in ("same", "no clear difference"):
                out["answer"] = "same"
            c = obj.get("confidence")
            out["confidence"] = int(c) if c is not None and str(c).strip().isdigit() else None
            out["reason"] = obj.get("reason")
            out["parse_ok"] = out["answer"] is not None
            if out["parse_ok"]:
                return out
        except Exception:
            pass
    m = _ANS_RE.search(text)
    if m:
        a = m.group(2).lower()
        out["answer"] = "same" if a.startswith(("same", "no")) else a
        out["parse_ok"] = True
    return out


# ----------------------------------------------------------------------------
# 7. Statistics helpers
# ----------------------------------------------------------------------------
def bootstrap_ci(values, groups=None, n_boot=10000, ci=95, seed=0, stat=np.mean):
    """Bootstrap CI of `stat`. If `groups` is given, resample groups (e.g., graph ids) with replacement."""
    rng = np.random.default_rng(seed)
    values = np.asarray(values, dtype=float)
    if groups is None:
        idx = rng.integers(0, len(values), size=(n_boot, len(values)))
        boots = np.array([stat(values[i]) for i in idx])
    else:
        groups = np.asarray(groups)
        uniq = np.unique(groups)
        members = {g: np.where(groups == g)[0] for g in uniq}
        boots = np.empty(n_boot)
        for b in range(n_boot):
            pick = rng.choice(uniq, size=len(uniq), replace=True)
            boots[b] = stat(values[np.concatenate([members[g] for g in pick])])
    lo, hi = np.percentile(boots, [(100 - ci) / 2, 100 - (100 - ci) / 2])
    return float(stat(values)), float(lo), float(hi)


Writing /kaggle/working/n07_output/pdcp_utils.py


## 2. Graphs and disjointness check against the N03 evaluation graphs

In [4]:
sys.path.insert(0, OUTPUT_DIR); import pdcp_utils as U; importlib.reload(U); print("utils", U.__version__)
def gseed(gi): return LJ_SEED + 1_000_000 * gi          # retries in make_graph add +1000*t, so seeds never collide
split_of = {gi: s for s, r in SPLIT.items() for gi in r}
def edge_key(G): return tuple(sorted(tuple(sorted(e)) for e in G.edges()))
eval_keys = {edge_key(U.make_graph(f, n, SEED + 100 * gi)) for f, n in FAMILY_SETS for gi in range(7)}
GRAPHS = [(f, n, gi, gseed(gi)) for f, n in FAMILY_SETS for gi in range(GRAPHS_PER_SET)]
# GRID is deterministic (one 7x7 grid): same graph as in evaluation by construction, but all drawings are new -> kept, reported.
dup = [g for g in GRAPHS if g[0] != "GRID" and edge_key(U.make_graph(g[0], g[1], g[3])) in eval_keys]
print(len(GRAPHS), "LiteJudge graphs | random-family graphs identical to an evaluation graph:", len(dup), "(GRID kept: deterministic)")
GRAPHS = [g for g in GRAPHS if g not in dup]

utils 0.1.3
240 LiteJudge graphs | random-family graphs identical to an evaluation graph: 0 (GRID kept: deterministic)


## 3. Base pools (two independent natural drawings per stress level per graph)

In [5]:
def make_base(fam, n, gi, gs, s, pool):
    import sys; sys.path.insert(0, OUTPUT_DIR)
    import torch, pdcp_utils as U; torch.set_num_threads(1)
    gc = U.GraphCache(U.make_graph(fam, n, gs))
    for attempt in range(4):
        r = U.pdsg_optimise(gc, {"ksm": s}, seed=gs % 10**6 + 7919 * pool + 101 * attempt + int(s * 1000))
        m = U.all_metrics(r.X, gc, with_realism=True)
        if r.converged and m["realistic"]: break
    return {"drawing_id": f"L{pool}_{fam}{n}_g{gi}_s{int(round(s*100))}", "role": f"base{pool}", "family": fam, "n": n, "graph": gi,
            "graph_seed": gs, "split": split_of[gi], "ok": bool(r.converged and m["realistic"]),
            **{k: m[k] for k in ["ksm", "nu", "eld", "xr", "mel", "aspect", "nn_max", "nn_min", "dense"]},
            "coords": r.X.ravel().round(6).tolist()}
jobs = [(f, n, gi, gs, s, pool) for f, n, gi, gs in GRAPHS for s in STRESS_LEVELS for pool in (1, 2)]
BASE = pd.DataFrame(Parallel(n_jobs=N_JOBS, verbose=2)(delayed(make_base)(*j) for j in jobs))
print("base ok rate", round(BASE.ok.mean(), 3), "|", len(BASE), "drawings |", f"{hours()*60:.0f} min")
BASE = BASE[BASE.ok].reset_index(drop=True)

[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  33 tasks      | elapsed:   20.9s
[Parallel(n_jobs=4)]: Done 154 tasks      | elapsed:   38.3s
[Parallel(n_jobs=4)]: Done 357 tasks      | elapsed: 15.4min
[Parallel(n_jobs=4)]: Done 640 tasks      | elapsed: 30.1min
[Parallel(n_jobs=4)]: Done 1005 tasks      | elapsed: 33.2min
[Parallel(n_jobs=4)]: Done 1450 tasks      | elapsed: 53.5min
[Parallel(n_jobs=4)]: Done 1977 tasks      | elapsed: 92.5min


base ok rate 0.969 | 2400 drawings | 96 min


[Parallel(n_jobs=4)]: Done 2400 out of 2400 | elapsed: 95.7min finished


## 4. Natural pairs (all pairs of natural drawings within a graph)

In [6]:
nat = []
for (fam, n, gi), g in BASE.groupby(["family", "n", "graph"]):
    recs = g.to_dict("records")
    for a, b in itertools.combinations(recs, 2):
        lo, hi = (a, b) if a["ksm"] <= b["ksm"] else (b, a)
        nat.append({"pair_id": f"N_{lo['drawing_id']}__{hi['drawing_id']}", "kind": "natural", "cond": "natural", "dstress_design": np.nan,
                    "family": fam, "n": n, "graph": gi, "split": lo["split"], "drawing_A": lo["drawing_id"], "drawing_B": hi["drawing_id"]})
NAT = pd.DataFrame(nat); print(len(NAT), "natural pairs")

10171 natural pairs


## 5. Decoupled PDSG pairs (four cues controlled relative to the partner)

In [7]:
Bi = BASE.set_index("drawing_id")
def gen_A(job):
    import sys, time; sys.path.insert(0, OUTPUT_DIR)
    import numpy as np, torch, pdcp_utils as U; torch.set_num_threads(1)
    gc = U.GraphCache(U.make_graph(job["family"], job["n"], job["graph_seed"]))
    tg = {"ksm": job["B_ksm"] - job["dstress"]}
    for p, c in zip(CUES, CONDS[job["cond"]]): tg[p] = job["B_" + p] + c * MARGIN[p]
    base = {k: job[k] for k in ["pair_id", "family", "n", "graph", "split", "dstress", "cond", "B_id"]}
    if min(tg[p] for p in CUES) < 0: return {**base, "ok": False}
    r0 = U.pdsg_optimise(gc, {"ksm": tg["ksm"]}, seed=job["seed"], max_steps=1500, max_restarts=1)
    r = U.pdsg_optimise(gc, tg, X0=r0.X, tol={"ksm": 0.004}, max_steps=1200, max_restarts=1, seed=job["seed"] + 1)
    m = U.all_metrics(r.X, gc, with_realism=True)
    return {**base, "ok": bool(r.converged and m["realistic"]), "graph_seed": job["graph_seed"],
            **{k: m[k] for k in ["ksm", "nu", "eld", "xr", "mel", "aspect", "nn_max", "nn_min", "dense"]}, "coords": r.X.ravel().round(6).tolist()}
rng = np.random.default_rng(LJ_SEED)
jobs = []
for (fam, n, gi), g in BASE[BASE.role == "base1"].groupby(["family", "n", "graph"]):
    for d in DSTRESS:
        cand = g[g.ksm - d >= STRESS_FLOOR]
        if not len(cand): continue
        for cond in CONDS:
            b = cand.iloc[int(rng.integers(len(cand)))]
            jobs.append({"pair_id": f"D_{fam}{n}_g{gi}_d{int(round(d*100)):02d}_{cond}", "family": fam, "n": int(n), "graph": int(gi),
                         "split": b.split, "graph_seed": int(b.graph_seed), "dstress": d, "cond": cond, "B_id": b.drawing_id,
                         **{"B_" + k: float(b[k]) for k in ["ksm"] + CUES}, "seed": int(rng.integers(1e8))})
print(len(jobs), "decoupled pair jobs")
res = []
for s in range(0, len(jobs), 500):
    if hours() > MAX_HOURS * 0.8: print("time budget reached"); break
    res += Parallel(n_jobs=N_JOBS)(delayed(gen_A)(j) for j in jobs[s:s + 500])
    print(f"{len(res)}/{len(jobs)} | ok {np.mean([r['ok'] for r in res]):.2f} | {hours()*60:.0f} min")
GEN = pd.DataFrame(res)
print("success by condition:", GEN.groupby("cond").ok.mean().round(2).to_dict())
print("success by dstress:", GEN.groupby("dstress").ok.mean().round(2).to_dict())
GOK = GEN[GEN.ok].copy(); GOK["drawing_id"] = "LA_" + GOK.pair_id.str[2:]
DEC = GOK.assign(kind="pdsg", drawing_A=GOK.drawing_id, drawing_B=GOK.B_id, dstress_design=GOK.dstress)[
    ["pair_id", "kind", "cond", "dstress_design", "family", "n", "graph", "split", "drawing_A", "drawing_B"]]

9380 decoupled pair jobs
500/9380 | ok 0.78 | 109 min
1000/9380 | ok 0.76 | 130 min
1500/9380 | ok 0.75 | 175 min
2000/9380 | ok 0.76 | 206 min
2500/9380 | ok 0.75 | 222 min
3000/9380 | ok 0.74 | 238 min
3500/9380 | ok 0.74 | 257 min
4000/9380 | ok 0.73 | 288 min
4500/9380 | ok 0.72 | 347 min
5000/9380 | ok 0.73 | 372 min
5500/9380 | ok 0.74 | 383 min
6000/9380 | ok 0.75 | 402 min
6500/9380 | ok 0.75 | 423 min
7000/9380 | ok 0.74 | 529 min
time budget reached
success by condition: {'conflict_all': 0.63, 'conflict_eld': 0.75, 'conflict_mel': 0.71, 'conflict_nu': 0.71, 'conflict_xr': 0.75, 'congruent': 0.83, 'neutral': 0.78}
success by dstress: {0.0: 0.96, 0.03: 0.96, 0.06: 0.98, 0.1: 0.81, 0.15: 0.36, 0.2: 0.02}


## 6. Assemble tables, labels and decorrelation check

In [8]:
MET = ["ksm", "nu", "eld", "xr", "mel", "aspect", "nn_max", "nn_min", "dense"]
DRW = pd.concat([BASE[["drawing_id", "role", "family", "n", "graph", "graph_seed", "split"] + MET + ["coords"]],
                 GOK.assign(role="A")[["drawing_id", "role", "family", "n", "graph", "graph_seed", "split"] + MET + ["coords"]]],
                ignore_index=True)
DRW["png"] = DRW.drawing_id.map(lambda d: f"images/{RES}/{d}.png")
PAIRS = pd.concat([NAT, DEC], ignore_index=True)
Di = DRW.set_index("drawing_id")
for k in ["ksm", "nu", "eld", "xr", "mel"]:
    PAIRS["A_" + k] = PAIRS.drawing_A.map(Di[k]).values; PAIRS["B_" + k] = PAIRS.drawing_B.map(Di[k]).values
    PAIRS["d_" + k] = PAIRS["B_" + k] - PAIRS["A_" + k]           # positive = B higher
PAIRS["lower"] = np.where(PAIRS.d_ksm.abs() < 0.005, "same", np.where(PAIRS.d_ksm > 0, "A", "B"))
def corr(df): return {p: round(float(pearsonr(df.d_ksm, df["d_" + p])[0]), 3) for p in CUES}
CORR = {"natural": corr(PAIRS[PAIRS.kind == "natural"]), "pdsg": corr(PAIRS[PAIRS.kind == "pdsg"])}
print(json.dumps(CORR))
print(PAIRS.groupby(["kind", "split"]).size().unstack())

{"natural": {"nu": 0.557, "eld": 0.352, "xr": 0.801, "mel": 0.76}, "pdsg": {"nu": 0.154, "eld": 0.156, "xr": 0.149, "mel": 0.176}}
split    test  train   val
kind                      
natural  1282   7573  1316
pdsg      621   3919   616


## 7. Render 384-px images

In [9]:
def render(rec):
    import sys; sys.path.insert(0, OUTPUT_DIR)
    import numpy as np, pdcp_utils as U
    gc = U.GraphCache(U.make_graph(rec["family"], rec["n"], rec["graph_seed"]))
    U.render_png(gc, np.array(rec["coords"]).reshape(-1, 2), f"{OUTPUT_DIR}/{rec['png']}", res=RES)
    return 1
Parallel(n_jobs=N_JOBS, verbose=1)(delayed(render)(r) for r in DRW.to_dict("records"))
DRW.to_parquet(f"{OUTPUT_DIR}/lj_drawings.parquet", index=False); PAIRS.to_parquet(f"{OUTPUT_DIR}/lj_pairs.parquet", index=False)
print(len(DRW), "drawings rendered |", f"{hours()*60:.0f} min")

[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    2.3s
[Parallel(n_jobs=4)]: Done 192 tasks      | elapsed:    9.3s
[Parallel(n_jobs=4)]: Done 442 tasks      | elapsed:   23.0s
[Parallel(n_jobs=4)]: Done 792 tasks      | elapsed:   38.1s
[Parallel(n_jobs=4)]: Done 1242 tasks      | elapsed:  1.0min
[Parallel(n_jobs=4)]: Done 1792 tasks      | elapsed:  1.5min
[Parallel(n_jobs=4)]: Done 2442 tasks      | elapsed:  2.0min
[Parallel(n_jobs=4)]: Done 3192 tasks      | elapsed:  2.6min
[Parallel(n_jobs=4)]: Done 4042 tasks      | elapsed:  3.3min
[Parallel(n_jobs=4)]: Done 4992 tasks      | elapsed:  4.1min
[Parallel(n_jobs=4)]: Done 6042 tasks      | elapsed:  5.0min
[Parallel(n_jobs=4)]: Done 7192 tasks      | elapsed:  5.9min
[Parallel(n_jobs=4)]: Done 7482 out of 7482 | elapsed:  6.2min finished


7482 drawings rendered | 536 min


## 8. Figures, report, manifest

In [10]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
M = np.array([[CORR[k][p] for p in CUES] for k in ["natural", "pdsg"]])
im = ax[0].imshow(M, cmap="RdBu_r", vmin=-1, vmax=1)
ax[0].set_xticks(range(4)); ax[0].set_xticklabels(["Δ NU", "Δ ELD", "Δ XR", "Δ MEL"]); ax[0].set_yticks(range(2)); ax[0].set_yticklabels(["natural pairs", "PDSG pairs"])
for i in range(2):
    for j in range(4): ax[0].text(j, i, f"{M[i,j]:+.2f}", ha="center", va="center")
ax[0].set_title("r(Δstress, Δcue)")
cc = PAIRS[PAIRS.kind == "pdsg"].groupby(["dstress_design", "cond"]).size().unstack(fill_value=0)
ax[1].imshow(cc.values, cmap="Greens"); ax[1].set_xticks(range(cc.shape[1])); ax[1].set_xticklabels(cc.columns, rotation=30, fontsize=7)
ax[1].set_yticks(range(cc.shape[0])); ax[1].set_yticklabels(cc.index)
for i in range(cc.shape[0]):
    for j in range(cc.shape[1]): ax[1].text(j, i, cc.values[i, j], ha="center", va="center", fontsize=7)
ax[1].set_title("PDSG pairs per Δstress × condition")
plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/fig_lj_data.png", dpi=130); plt.show()
REP = {"utils": U.__version__, "runtime_min": round(hours() * 60, 1), "graphs": len(GRAPHS), "removed_duplicate_graphs": len(dup),
       "drawings": len(DRW), "pairs": PAIRS.groupby(["kind", "split"]).size().unstack().to_dict(), "correlations": CORR,
       "pdsg_success_by_cond": GEN.groupby("cond").ok.mean().round(3).to_dict(),
       "pdsg_success_by_dstress": GEN.groupby("dstress").ok.mean().round(3).to_dict(),
       "label_counts": PAIRS.groupby(["kind", "lower"]).size().unstack().to_dict(), "margins": MARGIN}
json.dump(REP, open(f"{OUTPUT_DIR}/n07_report.json", "w"), indent=2, default=str)
shutil.rmtree(f"{OUTPUT_DIR}/__pycache__", ignore_errors=True)
lines = [f"N07 LiteJudge data · {REP['runtime_min']} min · {len(DRW)} drawings · {len(PAIRS)} pairs", f"corr: {json.dumps(CORR)}", ""]
for f in sorted(os.listdir(OUTPUT_DIR)):
    p = f"{OUTPUT_DIR}/{f}"
    if os.path.isfile(p): lines.append(f"{f:28s} {os.path.getsize(p)/1024:10.1f} KB")
lines.append(f"images/{RES}/: {len(os.listdir(f'{OUTPUT_DIR}/images/{RES}'))} PNG")
open(f"{OUTPUT_DIR}/MANIFEST.txt", "w").write("\n".join(lines)); print("\n".join(lines))

N07 LiteJudge data · 535.7 min · 7482 drawings · 15327 pairs
corr: {"natural": {"nu": 0.557, "eld": 0.352, "xr": 0.801, "mel": 0.76}, "pdsg": {"nu": 0.154, "eld": 0.156, "xr": 0.149, "mel": 0.176}}

fig_lj_data.png                    54.0 KB
lj_drawings.parquet              7853.2 KB
lj_pairs.parquet                 1282.1 KB
n07_report.json                     1.1 KB
pdcp_utils.py                      22.4 KB
images/384/: 7482 PNG
